# 循环执行：频率提取与判断

**用途**：在频率计算流程中循环调用，仅提取频率、打印状态，用于判断 PASS / HAS_IMAG / INCOMPLETE。

In [ ]:
import os
import re
from pathlib import Path

In [ ]:
# 从 config 读取
import sys
_p = Path.cwd()
while _p != _p.parent:
    if (_p / "shared" / "config.py").exists():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
from shared.load_config import load_config
cfg = load_config()
zpe_root = cfg.ZPE_ROOT

In [ ]:
def read_frequencies(outcar_path):
    """从 OUTCAR 读取频率，返回 {frequencies, imaginary_freqs, real_freqs, zpe, n_modes} 或 None"""
    if not os.path.exists(outcar_path):
        return None
    freq_pattern = r'(-?\d+\.?\d*(?:[eE][+-]?\d+)?)\s+cm-1'
    all_freqs, imag_freqs, real_freqs = [], [], []
    with open(outcar_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if "THz" in line and ("cm^-1" in line or "cm-1" in line):
                for m in re.findall(freq_pattern, line):
                    try:
                        freq = float(m)
                        is_imag = (freq < 0) or "f/i" in line
                        if is_imag:
                            imag_freqs.append(freq)
                            all_freqs.append(abs(freq))
                        else:
                            real_freqs.append(freq)
                            all_freqs.append(freq)
                    except ValueError:
                        pass
    if not all_freqs:
        return None
    seen = {}
    for f in all_freqs:
        k = round(f, 2)
        if k not in seen:
            seen[k] = f
    all_freqs = list(seen.values())
    zpe = sum(all_freqs) / 2000.0 if all_freqs else None
    return {'frequencies': all_freqs, 'imaginary_freqs': imag_freqs, 'real_freqs': real_freqs, 'zpe': zpe, 'n_modes': len(all_freqs)}


def scan_zpe_job_dirs(zpe_root):
    """扫描含 OUTCAR 的任务目录"""
    base = Path(zpe_root).resolve()
    return sorted(set(str(p.parent) for p in base.rglob("OUTCAR") if p.parent.resolve() != base))


def status_from_freq(freq_data):
    """PASS / HAS_IMAG / INCOMPLETE"""
    if not freq_data:
        return "INCOMPLETE"
    if freq_data.get('imaginary_freqs'):
        return "HAS_IMAG"
    return "PASS"

In [ ]:
# 扫描并打印频率状态
job_dirs = scan_zpe_job_dirs(zpe_root)
print(f"📌 ZPE: {zpe_root}")
print(f"找到 {len(job_dirs)} 个任务目录\n")
print("=" * 60)

for jd in job_dirs:
    rel = os.path.relpath(jd, zpe_root)
    outcar = os.path.join(jd, "OUTCAR")
    fd = read_frequencies(outcar)
    st = status_from_freq(fd)
    print(f"{rel}")
    print(f"  状态: {st}")
    if fd:
        if fd.get('imaginary_freqs'):
            print(f"  虚频: {[f'{f:.1f}' for f in fd['imaginary_freqs']]} cm^-1")
        if fd.get('zpe'):
            print(f"  ZPE: {fd['zpe']:.6f} eV")
    else:
        print(f"  (无 OUTCAR 或未完成)")
    print()
print("=" * 60)
print("PASS=无虚频 | HAS_IMAG=有虚频→进入507 | INCOMPLETE=未完成")